## ***Initials***

In [1]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import json
from tqdm import tqdm
import numpy as np
import ollama

## ***Help Functions***

In [5]:
# === Embed function ===
def get_embedding(text, tokenizer, model, device):
    prompt = "Represent this sentence for retrieval: " + text
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, padding=True).to(device)
    with torch.no_grad():
        output = model(**inputs)
        embedding = output.last_hidden_state[:, 0]
        embedding = F.normalize(embedding, p=2, dim=1)
    return embedding.cpu().numpy()[0]

## ***Load the Data***

In [7]:
# === Load the data ===
df_naces = pd.read_csv("C:\\Users\\Giorgos\\Desktop\\HMMY\\10ο Εξάμηνο\\Διπλωματική\\2. General Data - Directories\\3. NACE Data\\unique_naces.csv")
nace_dict = dict(zip(df_naces['NACE Code'].astype(str), df_naces['Class Description']))
print(len(df_naces))
df_naces.head()

307


,NACE Code,Class Description
0,1.13,"Growing of vegetables and melons, roots and tu..."
1,1.30,Plant propagation
2,1.49,Raising of other animals
3,1.61,Support activities for crop production
4,2.40,Support services to forestry


## ***Load the Embedder Model***

In [2]:
# Load the Embedder and its tokenizer
classifier_model_name = "BAAI/bge-large-en-v1.5"

In [3]:
classifier_tokenizer = AutoTokenizer.from_pretrained(classifier_model_name)
classifier_model = AutoModel.from_pretrained(classifier_model_name)

In [4]:
# Set device (CPU or CUDA if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
classifier_model.to(device)

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 1024, padding_idx=0)
    (position_embeddings): Embedding(512, 1024)
    (token_type_embeddings): Embedding(2, 1024)
    (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-23): 24 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=1024, out_features=1024, bias=True)
            (key): Linear(in_features=1024, out_features=1024, bias=True)
            (value): Linear(in_features=1024, out_features=1024, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=1024, out_features=1024, bias=True)
            (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, 

## ***Embed all the NACE Descriptions***

In [11]:
# Precompute NACE embeddings
nace_embeddings = []
for idx, row in df_naces.iterrows():
    emb = get_embedding(row["Class Description"], classifier_tokenizer, classifier_model, device)
    nace_embeddings.append({
        "nace_code": row["NACE Code"],
        "embedding": emb.tolist()
    })

## ***Save the Data***

In [13]:
# Save to JSON
with open("C:\\Users\\Giorgos\\Desktop\\HMMY\\10ο Εξάμηνο\\Διπλωματική\\8. LLM - RAG\\Extracted Files\\nace_descriptions_embeddings.json", "w", encoding="utf-8") as f:
    json.dump(nace_embeddings, f, ensure_ascii=False)